## Imports

In [1]:
import spacy
import nltk

In [2]:
# Baixar os recursos do NLTK
try:
    nltk.data.find('tokenizers/punkt_tab')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('stemmers/rslp')
except LookupError:
    nltk.download('punkt_tab')
    nltk.download('stopwords')
    nltk.download('rslp')

In [3]:
# Baixar os recursos do Spacy

try:
    nlp = spacy.load("pt_core_news_sm")
    print("Modelo carregado com sucesso")
except OSError:
    print("Executar o seguinte comando: python -m spacy download pt_core_news_sm")

Modelo carregado com sucesso


In [4]:
# Texto com ruído: Pontuação, Case variado, Dados sensíveis e formatação um pouco inconsistente
texto_bruto = """
    URGENTE: O Ciêntista de dados (Senior) enviou um e-mail para rh@empresa.com.br!!!
    O salário proposto é de R$ 12.500,00.
    O CPF do candidato é 123.456.789-00 e ele domina: Python, NLP e REGEX.
    OBS: Nós amamos processamento de linguagem natural...
"""

print(texto_bruto)


    URGENTE: O Ciêntista de dados (Senior) enviou um e-mail para rh@empresa.com.br!!!
    O salário proposto é de R$ 12.500,00.
    O CPF do candidato é 123.456.789-00 e ele domina: Python, NLP e REGEX.
    OBS: Nós amamos processamento de linguagem natural...



## Tokenização

In [5]:
# Abordagem mais ingênua (ruim para NLP)
tokens_split = texto_bruto.split()
tokens_split[:5]

['URGENTE:', 'O', 'Ciêntista', 'de', 'dados']

In [6]:
tokens_nlk = nltk.word_tokenize(texto_bruto,language="portuguese")
tokens_nlk[:5]

['URGENTE', ':', 'O', 'Ciêntista', 'de']

In [7]:
doc = nlp(texto_bruto)

tokens_spacy = [token.text for token in doc]
tokens_spacy[:5]

['\n    ', 'URGENTE', ':', 'O', 'Ciêntista']

In [8]:
# Abordagem de NLTK (Tab)

tab_tokens = nltk.tokenize.TabTokenizer().tokenize(texto_bruto)
tab_tokens

['\n    URGENTE: O Ciêntista de dados (Senior) enviou um e-mail para rh@empresa.com.br!!!\n    O salário proposto é de R$ 12.500,00.\n    O CPF do candidato é 123.456.789-00 e ele domina: Python, NLP e REGEX.\n    OBS: Nós amamos processamento de linguagem natural...\n']

In [9]:
line_tokens = nltk.tokenize.line_tokenize(texto_bruto)
line_tokens

['    URGENTE: O Ciêntista de dados (Senior) enviou um e-mail para rh@empresa.com.br!!!',
 '    O salário proposto é de R$ 12.500,00.',
 '    O CPF do candidato é 123.456.789-00 e ele domina: Python, NLP e REGEX.',
 '    OBS: Nós amamos processamento de linguagem natural...']

In [10]:
# Abordagem Char Tokenizer

char_tokens = [char for char in texto_bruto]
char_tokens[:10]

['\n', ' ', ' ', ' ', ' ', 'U', 'R', 'G', 'E', 'N']

## Normalização de texto

In [11]:
# Normalizar o texto segundo o Case

texto_lower = texto_bruto.lower()
print(texto_lower)


    urgente: o ciêntista de dados (senior) enviou um e-mail para rh@empresa.com.br!!!
    o salário proposto é de r$ 12.500,00.
    o cpf do candidato é 123.456.789-00 e ele domina: python, nlp e regex.
    obs: nós amamos processamento de linguagem natural...



In [12]:
# Removendo a pontuação do texto
pontuacao = """!()-[]{};:',<>./?@#$%^&*_~"""
texto_sem_pontuacao = ''.join(char for char in texto_lower if char not in pontuacao)
texto_sem_pontuacao

'\n    urgente o ciêntista de dados senior enviou um email para rhempresacombr\n    o salário proposto é de r 1250000\n    o cpf do candidato é 12345678900 e ele domina python nlp e regex\n    obs nós amamos processamento de linguagem natural\n'

In [13]:
# Remover acentos
import unicodedata
texto_sem_acento = ''.join(char for char in unicodedata.normalize('NFD', texto_sem_pontuacao) if unicodedata.category(char) != 'Mn')
texto_sem_acento

'\n    urgente o cientista de dados senior enviou um email para rhempresacombr\n    o salario proposto e de r 1250000\n    o cpf do candidato e 12345678900 e ele domina python nlp e regex\n    obs nos amamos processamento de linguagem natural\n'

In [14]:
def preprocess_text(text):
    texto_lower = text.lower()
    texto_sem_pontuacao = ''.join(char for char in texto_lower if char not in pontuacao)
    texto_sem_acento = ''.join(char for char in unicodedata.normalize('NFD', texto_sem_pontuacao) if unicodedata.category(char) != 'Mn')
    return texto_sem_acento


In [15]:
text_normalize = preprocess_text(texto_bruto)
text_normalize

'\n    urgente o cientista de dados senior enviou um email para rhempresacombr\n    o salario proposto e de r 1250000\n    o cpf do candidato e 12345678900 e ele domina python nlp e regex\n    obs nos amamos processamento de linguagem natural\n'

## Stop Words e Pontuação

In [16]:
# Carregar uma lista de stop words

stop_words_pt = nltk.corpus.stopwords.words('portuguese')

len(stop_words_pt)

207

In [17]:
# Filtragem de stopwords e alfanumericos

tokens_limpos = [token for token in tokens_nlk if token.isalpha() and token.lower() not in stop_words_pt]
tokens_limpos[:5]

['URGENTE', 'Ciêntista', 'dados', 'Senior', 'enviou']

In [18]:
len(tokens_limpos)

20

## Stemming

In [19]:
# Inicializar o Stemmer do NLTK
stemmer = nltk.stem.RSLPStemmer()

doc = nlp(texto_bruto)

for token in doc:
    if token.is_punct or token.is_space:
        continue

    word = token.text.lower()
    stem = stemmer.stem(word)

    print(f"word: {word} -> {stem}")

word: urgente -> urgent
word: o -> o
word: ciêntista -> ciênt
word: de -> de
word: dados -> dad
word: senior -> seni
word: enviou -> envi
word: um -> um
word: e-mail -> e-mail
word: para -> par
word: rh@empresa.com.br -> rh@empresa.com.br
word: o -> o
word: salário -> salári
word: proposto -> propost
word: é -> é
word: de -> de
word: r$ -> r$
word: 12.500,00 -> 12.500,00
word: o -> o
word: cpf -> cpf
word: do -> do
word: candidato -> candidat
word: é -> é
word: 123.456. -> 123.456.
word: 789-00 -> 789-00
word: e -> e
word: ele -> ele
word: domina -> domin
word: python -> python
word: nlp -> nlp
word: e -> e
word: regex -> regex
word: obs -> ob
word: nós -> nó
word: amamos -> am
word: processamento -> process
word: de -> de
word: linguagem -> lingu
word: natural -> natur


## Lemantização

In [20]:
for token in doc:
    if token.is_punct or token.is_space:
        continue

    word = token.text.lower()
    
    lemma = token.lemma_
    pos = token.pos_

    print(f"word: {word} -> {lemma} -> {pos}")

word: urgente -> URGENTE -> NOUN
word: o -> o -> DET
word: ciêntista -> Ciêntista -> PROPN
word: de -> de -> ADP
word: dados -> dado -> NOUN
word: senior -> Senior -> PROPN
word: enviou -> enviar -> VERB
word: um -> um -> DET
word: e-mail -> e-mail -> NOUN
word: para -> para -> ADP
word: rh@empresa.com.br -> rh@empresa.com.br -> NOUN
word: o -> o -> DET
word: salário -> salário -> NOUN
word: proposto -> propor -> VERB
word: é -> ser -> AUX
word: de -> de -> ADP
word: r$ -> R$ -> SYM
word: 12.500,00 -> 12.500,00 -> NUM
word: o -> o -> DET
word: cpf -> CPF -> NOUN
word: do -> de o -> ADP
word: candidato -> candidato -> NOUN
word: é -> ser -> AUX
word: 123.456. -> 123.456. -> ADJ
word: 789-00 -> 789-00 -> NUM
word: e -> e -> CCONJ
word: ele -> ele -> PRON
word: domina -> dominar -> VERB
word: python -> Python -> PROPN
word: nlp -> NLP -> PROPN
word: e -> e -> CCONJ
word: regex -> REGEX -> PROPN
word: obs -> OBS -> PROPN
word: nós -> nós -> PRON
word: amamos -> amar -> VERB
word: processam

### Comparar Stemming e Lemantização

In [21]:
data_comparacao = []

for token in doc:
    if token.is_punct or token.is_space:
        continue

    word = token.text.lower()
    stem = stemmer.stem(word)
    lemma = token.lemma_
    pos = token.pos_

    data_comparacao.append([word,lemma,stem,pos])


In [22]:
import pandas as pd

df = pd.DataFrame(data_comparacao, columns=["Original", "Lemma", "Stem", "POS"])

df_filtrado = df[df['Stem'] != df['Lemma']]

df_filtrado.head(20)

,Original,Lemma,Stem,POS
0,urgente,URGENTE,urgent,NOUN
2,ciêntista,Ciêntista,ciênt,PROPN
4,dados,dado,dad,NOUN
5,senior,Senior,seni,PROPN
6,enviou,enviar,envi,VERB
9,para,para,par,ADP
12,salário,salário,salári,NOUN
13,proposto,propor,propost,VERB
14,é,ser,é,AUX
16,r$,R$,r$,SYM
